In [2]:
!pip install transformers

  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.4 MB 2.5 MB/s eta 0:00:04
   ----- -------------

In [9]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

from transformers import pipeline, AutoConfig
from clean_social.evaluation.metrics import compute_classification_metrics, LABEL_ORDER

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
TEST_SIZE = 0.2

#Required SOTA model from your instruction
SOTA_MODEL_NAME = 'siberett/roberta-sentiment-analysis-finetune'

#Project paths (assumes notebook is run from clean_social_solution root)
ROOT = Path('../../').resolve()
ARTIFACTS = ROOT / 'artifacts'
REPORTS = ROOT / 'reports'
TASK5_DIR = REPORTS / 'task5'
PLOTS_DIR = TASK5_DIR / 'plots'
TASK5_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT)
print('Task5 output dir:', TASK5_DIR)

Root: C:\Users\Fares\Documents\university\CleanSocial\clean_social_solution
Task5 output dir: C:\Users\Fares\Documents\university\CleanSocial\clean_social_solution\reports\task5


In [10]:
opt_results_path = REPORTS / 'optimization_results.csv'
opt_details_path = REPORTS / 'optimization_results_detailed.json'

opt_df = pd.read_csv(opt_results_path)
best_row = opt_df.sort_values('tuned_f1', ascending=False).iloc[0]

best_scheme = best_row['scheme'] # s1, s2, or s3
best_repr = best_row['representation'] # tfidf or glove
best_model_key = f'{best_scheme}_{best_repr}_svc'
best_model_path = ARTIFACTS / 'models' / 'optimized' / f'{best_model_key}.joblib'

with open(opt_details_path, 'r', encoding='utf-8') as f:
    opt_details = json.load(f)

print('Best optimized model key:', best_model_key)
print('Best scheme:', best_scheme)
print('Best representation:', best_repr)
print('Best tuned F1:', float(best_row['tuned_f1']))
print('Best model path exists:', best_model_path.exists())

Best optimized model key: s3_tfidf_svc
Best scheme: s3
Best representation: tfidf
Best tuned F1: 0.6962833914053427
Best model path exists: True


In [12]:
labels_df = pd.read_csv(ARTIFACTS / 'features' / 'labels_400.csv')

scheme_to_processed = {
's1': ROOT / 'data' / 'processed' / 's1_minimal.csv',
's2': ROOT / 'data' / 'processed' / 's2_standard.csv',
's3': ROOT / 'data' / 'processed' / 's3_extended.csv',
}

processed_df = pd.read_csv(scheme_to_processed[best_scheme]).reset_index(names='source_index')

#Keep category and content for analysis
text_cat_df = processed_df[['source_index', 'content', 'category']].copy()

#Merge labels with text/category
base_df = labels_df.merge(text_cat_df, on='source_index', how='inner')

#Load matching feature matrix for the optimized model
feat_path = ARTIFACTS / 'features' / f'{best_repr}_{best_scheme}.csv'
feat_df = pd.read_csv(feat_path)

#Ensure correct alignment by record_id
full_df = base_df.merge(feat_df, on='record_id', how='inner').copy()
full_df['content'] = full_df['content'].fillna('').astype(str)
full_df['category'] = full_df['category'].fillna('other').astype(str)
full_df['ground_truth'] = full_df['ground_truth'].astype(str)

feature_cols = [c for c in feat_df.columns if c != 'record_id']
X_all = full_df[feature_cols].to_numpy(dtype=float)
y_all = full_df['ground_truth'].tolist()
record_ids_all = full_df['record_id'].tolist()

print('Merged rows:', len(full_df))
print('Class distribution:\n', full_df['ground_truth'].value_counts())
print('Category distribution (top 10):\n', full_df['category'].value_counts().head(10))

KeyError: 'category'

In [ ]:
X_train, X_test, y_train, y_test, rid_train, rid_test = train_test_split(
X_all,
y_all,
record_ids_all,
test_size=TEST_SIZE,
random_state=RANDOM_SEED,
stratify=y_all
)

test_id_set = set(rid_test)
test_df = full_df[full_df['record_id'].isin(test_id_set)].copy()

#Keep test_df in same order as rid_test for exact alignment
rid_to_row = {int(r): i for i, r in enumerate(test_df['record_id'].tolist())}
test_df = test_df.iloc[[rid_to_row[int(r)] for r in rid_test]].reset_index(drop=True)

print('Train size:', len(X_train))
print('Test size:', len(X_test))
print('Test class distribution:\n', pd.Series(y_test).value_counts())

In [ ]:
optimized_model = joblib.load(best_model_path)

#Predictions on all records
opt_pred_all = optimized_model.predict(X_all).tolist()
opt_prob_all = optimized_model.predict_proba(X_all)

#Predictions on test set
opt_pred_test = optimized_model.predict(X_test).tolist()
opt_prob_test = optimized_model.predict_proba(X_test)

opt_metrics = compute_classification_metrics(
y_true=y_test,
y_pred=opt_pred_test,
score_matrix=opt_prob_test
)

print('Optimized model metrics (test):')
print(json.dumps(opt_metrics, indent=2))

In [ ]:
def normalize_model_label(label_str):
    s = str(label_str).strip().lower()

    # semantic names
    if 'neg' in s:
        return 'Negative'
    if 'neu' in s:
        return 'Neutral'
    if 'pos' in s:
        return 'Positive'

    # numeric stars/ratings labels (if any)
    digits = ''.join(ch for ch in s if ch.isdigit())
    if digits:
        try:
            v = int(digits)
            if v <= 2:
                return 'Negative'
            elif v == 3:
                return 'Neutral'
            else:
                return 'Positive'
        except Exception:
            pass

    return None

def map_scores_to_three_class(score_list, id2label=None, neutral_conf_threshold=0.60):
    """
    score_list is list of dicts, e.g.:
    [{'label': 'LABEL_0', 'score': ...}, {'label': 'LABEL_1', 'score': ...}, ...]
    Returns:
      pred_label, prob_vector(np.array of size 3 in LABEL_ORDER order), mapping_note
    """
    # First pass: semantic mapping from labels
    probs = {'Negative': 0.0, 'Neutral': 0.0, 'Positive': 0.0}
    unresolved = []

    for item in score_list:
        raw_label = str(item['label'])
        score = float(item['score'])

        mapped = normalize_model_label(raw_label)
        if mapped is not None:
            probs[mapped] += score
        else:
            unresolved.append((raw_label, score))

    # If unresolved exist, try id-based fallback
    if unresolved:
        # parse LABEL_0 style into index
        temp = []
        for raw_label, score in unresolved:
            lower = raw_label.lower()
            if 'label_' in lower:
                try:
                    idx = int(lower.split('label_')[-1])
                    temp.append((idx, score))
                except Exception:
                    pass

        # Common fallback for 3-class heads: 0 neg, 1 neu, 2 pos
        if len(score_list) == 3 and len(temp) >= 1:
            for idx, score in temp:
                if idx == 0:
                    probs['Negative'] += score
                elif idx == 1:
                    probs['Neutral'] += score
                elif idx == 2:
                    probs['Positive'] += score

        # Binary fallback with confidence-to-neutral transformation
        elif len(score_list) == 2 and len(temp) >= 1:
            neg_score = 0.0
            pos_score = 0.0
            for idx, score in temp:
                if idx == 0:
                    neg_score = max(neg_score, score)
                elif idx == 1:
                    pos_score = max(pos_score, score)

            max_bin = max(neg_score, pos_score)
            if max_bin < neutral_conf_threshold:
                # uncertain binary prediction -> convert uncertainty to Neutral mass
                neutral_mass = 1.0 - max_bin
                rem = 1.0 - neutral_mass
                probs['Negative'] += rem * (neg_score / max(neg_score + pos_score, 1e-12))
                probs['Positive'] += rem * (pos_score / max(neg_score + pos_score, 1e-12))
                probs['Neutral'] += neutral_mass
            else:
                probs['Negative'] += neg_score
                probs['Positive'] += pos_score

    vec = np.array([probs['Negative'], probs['Neutral'], probs['Positive']], dtype=float)

    # normalize for safety
    s = vec.sum()
    if s <= 0:
        vec = np.array([1/3, 1/3, 1/3], dtype=float)
    else:
        vec = vec / s

    pred = LABEL_ORDER[int(np.argmax(vec))]
    return pred, vec

# Build HF pipeline
sota_pipe = pipeline(
    task='text-classification',
    model=SOTA_MODEL_NAME,
    top_k=None,
    truncation=True,
    max_length=256
)

# Capture model metadata for notebook documentation requirement
sota_cfg = AutoConfig.from_pretrained(SOTA_MODEL_NAME)
model_metadata = {
    'model_name': SOTA_MODEL_NAME,
    'architectures': getattr(sota_cfg, 'architectures', None),
    'num_labels': int(getattr(sota_cfg, 'num_labels', 0)),
    'id2label': {int(k): v for k, v in dict(getattr(sota_cfg, 'id2label', {})).items()} if getattr(sota_cfg, 'id2label', None) else None,
    'label2id': dict(getattr(sota_cfg, 'label2id', {})) if getattr(sota_cfg, 'label2id', None) else None,
    'revision_or_commit': getattr(sota_cfg, '_commit_hash', None)
}

print('SOTA model metadata:')
print(json.dumps(model_metadata, indent=2, default=str))

# Inference on all labeled records (required)
all_texts = full_df['content'].tolist()
sota_raw_all = sota_pipe(all_texts, batch_size=32)

sota_pred_all = []
sota_prob_all = []
for scores in sota_raw_all:
    pred, vec = map_scores_to_three_class(scores)
    sota_pred_all.append(pred)
    sota_prob_all.append(vec)

sota_prob_all = np.vstack(sota_prob_all)

# Build test set arrays aligned with test order
rid_to_index_all = {int(r): i for i, r in enumerate(full_df['record_id'].tolist())}
test_indices_in_all = [rid_to_index_all[int(r)] for r in rid_test]

sota_pred_test = [sota_pred_all[i] for i in test_indices_in_all]
sota_prob_test = sota_prob_all[test_indices_in_all, :]

sota_metrics = compute_classification_metrics(
    y_true=y_test,
    y_pred=sota_pred_test,
    score_matrix=sota_prob_test
)

print('SOTA model metrics (test):')
print(json.dumps(sota_metrics, indent=2))


: 

In [ ]:
pred_all_df = full_df[['record_id', 'source_index', 'content', 'category', 'ground_truth']].copy()
pred_all_df['optimized_pred'] = opt_pred_all
pred_all_df['sota_pred'] = sota_pred_all
pred_all_df['optimized_correct'] = (pred_all_df['optimized_pred'] == pred_all_df['ground_truth']).astype(int)
pred_all_df['sota_correct'] = (pred_all_df['sota_pred'] == pred_all_df['ground_truth']).astype(int)

# Add model probability columns
for j, lbl in enumerate(LABEL_ORDER):
    pred_all_df[f'optimized_prob_{lbl.lower()}'] = opt_prob_all[:, j]
    pred_all_df[f'sota_prob_{lbl.lower()}'] = sota_prob_all[:, j]

pred_all_out = TASK5_DIR / 'predictions_all_records.csv'
pred_all_df.to_csv(pred_all_out, index=False)

test_pred_df = pred_all_df[pred_all_df['record_id'].isin(rid_test)].copy()
test_pred_df = test_pred_df.set_index('record_id').loc[rid_test].reset_index()
test_out = TASK5_DIR / 'predictions_test.csv'
test_pred_df.to_csv(test_out, index=False)

print('Saved:', pred_all_out)
print('Saved:', test_out)


: 

In [ ]:
def flatten_metrics_row(model_name, metrics_dict):
    row = {
        'model': model_name,
        'accuracy': metrics_dict.get('accuracy'),
        'precision_macro': metrics_dict.get('precision_macro'),
        'recall_macro': metrics_dict.get('recall_macro'),
        'f1_macro': metrics_dict.get('f1_macro'),
        'roc_auc_ovr': metrics_dict.get('roc_auc_ovr'),
        'confusion_matrix': json.dumps(metrics_dict.get('confusion_matrix'))
    }
    return row

summary_df = pd.DataFrame([
    flatten_metrics_row(f'Optimized ({best_model_key})', opt_metrics),
    flatten_metrics_row(f'SOTA ({SOTA_MODEL_NAME})', sota_metrics),
])

summary_out = TASK5_DIR / 'model_comparison_summary.csv'
summary_df.to_csv(summary_out, index=False)

print(summary_df)
print('Saved:', summary_out)


In [ ]:
def category_metrics(df, true_col, pred_col):
    rows = []
    for cat, g in df.groupby('category'):
        y_t = g[true_col].tolist()
        y_p = g[pred_col].tolist()
        m = compute_classification_metrics(y_t, y_p, score_matrix=None)
        rows.append({
            'category': cat,
            'n_samples': len(g),
            'accuracy': m['accuracy'],
            'f1_macro': m['f1_macro'],
        })
    return pd.DataFrame(rows)

cat_opt = category_metrics(test_pred_df, 'ground_truth', 'optimized_pred').rename(
    columns={'accuracy': 'optimized_acc', 'f1_macro': 'optimized_f1'}
)
cat_sota = category_metrics(test_pred_df, 'ground_truth', 'sota_pred').rename(
    columns={'accuracy': 'sota_acc', 'f1_macro': 'sota_f1'}
)

cat_cmp = cat_opt.merge(cat_sota[['category', 'sota_acc', 'sota_f1']], on='category', how='inner')
cat_cmp = cat_cmp.sort_values('n_samples', ascending=False).reset_index(drop=True)

cat_out = TASK5_DIR / 'per_category_comparison.csv'
cat_cmp.to_csv(cat_out, index=False)

print(cat_cmp.head(20))
print('Saved:', cat_out)


In [ ]:
import seaborn as sns

sns.set_theme(style='whitegrid')

# Plot 1: Overall metric comparison
metric_cols = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr']
plot_df = summary_df[['model'] + metric_cols].melt(id_vars='model', var_name='metric', value_name='value')

plt.figure(figsize=(11, 5))
sns.barplot(data=plot_df, x='metric', y='value', hue='model')
plt.ylim(0, 1)
plt.title('Optimized vs SOTA - Overall Metrics')
plt.xlabel('Metric')
plt.ylabel('Score')
plt.xticks(rotation=20)
plt.tight_layout()
p1 = PLOTS_DIR / 'overall_metrics_comparison.png'
plt.savefig(p1, dpi=220)
plt.show()

# Plot 2: Per-category Accuracy
cat_acc_plot = cat_cmp.melt(
    id_vars=['category', 'n_samples'],
    value_vars=['optimized_acc', 'sota_acc'],
    var_name='model_metric',
    value_name='score'
)

plt.figure(figsize=(12, 6))
sns.barplot(data=cat_acc_plot, x='category', y='score', hue='model_metric')
plt.ylim(0, 1)
plt.title('Per-Category Accuracy: Optimized vs SOTA')
plt.xlabel('Category')
plt.ylabel('Accuracy')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
p2 = PLOTS_DIR / 'per_category_accuracy.png'
plt.savefig(p2, dpi=220)
plt.show()

# Plot 3: Per-category Macro-F1
cat_f1_plot = cat_cmp.melt(
    id_vars=['category', 'n_samples'],
    value_vars=['optimized_f1', 'sota_f1'],
    var_name='model_metric',
    value_name='score'
)

plt.figure(figsize=(12, 6))
sns.barplot(data=cat_f1_plot, x='category', y='score', hue='model_metric')
plt.ylim(0, 1)
plt.title('Per-Category Macro-F1: Optimized vs SOTA')
plt.xlabel('Category')
plt.ylabel('Macro-F1')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
p3 = PLOTS_DIR / 'per_category_f1.png'
plt.savefig(p3, dpi=220)
plt.show()

# Plot 4: Confusion matrices side-by-side
cm_opt = confusion_matrix(y_test, opt_pred_test, labels=LABEL_ORDER)
cm_sota = confusion_matrix(y_test, sota_pred_test, labels=LABEL_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues', xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER, ax=axes[0])
axes[0].set_title(f'Optimized Confusion Matrix ({best_model_key})')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_sota, annot=True, fmt='d', cmap='Oranges', xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER, ax=axes[1])
axes[1].set_title(f'SOTA Confusion Matrix ({SOTA_MODEL_NAME})')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
p4 = PLOTS_DIR / 'confusion_matrices_side_by_side.png'
plt.savefig(p4, dpi=220)
plt.show()

# Plot 5: Category vs error-rate heatmap for both models
err_df = test_pred_df.copy()
err_df['optimized_error'] = (err_df['optimized_pred'] != err_df['ground_truth']).astype(float)
err_df['sota_error'] = (err_df['sota_pred'] != err_df['ground_truth']).astype(float)

err_cat = err_df.groupby('category')[['optimized_error', 'sota_error']].mean().sort_values('sota_error', ascending=False)

plt.figure(figsize=(8, max(4, 0.35 * len(err_cat))))
sns.heatmap(err_cat, annot=True, fmt='.2f', cmap='Reds', cbar_kws={'label': 'Error Rate'})
plt.title('Error Rate by Category')
plt.xlabel('Model')
plt.ylabel('Category')
plt.tight_layout()
p5 = PLOTS_DIR / 'error_rate_by_category_heatmap.png'
plt.savefig(p5, dpi=220)
plt.show()

print('Saved plots:')
for p in [p1, p2, p3, p4, p5]:
    print('-', p)



In [ ]:
best_cat_gain = cat_cmp.assign(f1_gain=cat_cmp['sota_f1'] - cat_cmp['optimized_f1']).sort_values('f1_gain', ascending=False)
worst_cat_gain = cat_cmp.assign(f1_gain=cat_cmp['sota_f1'] - cat_cmp['optimized_f1']).sort_values('f1_gain', ascending=True)

report_md = f"""
# Task 5 Results Report

## 1. Introduction
This report benchmarks the optimized sentiment model from Task 4 against a Hugging Face SOTA model on the same labeled dataset.
The comparison is performed on an identical test split and includes per-category analysis to assess category-specific strengths/weaknesses.

## 2. Methodology
- Optimized model selected from Task 4 results: {best_model_key}
- SOTA model used: {SOTA_MODEL_NAME}
- Evaluation labels: {LABEL_ORDER}
- Test split settings: random_state={RANDOM_SEED}, test_size={TEST_SIZE}, stratified by ground truth
- SOTA inference done with transformers pipeline('text-classification', model=...)
- Label mapping logic:
  - semantic mapping for labels containing neg/neu/pos
  - numeric-label mapping for star-like outputs
  - binary fallback with confidence-to-neutral conversion (threshold=0.60) when needed
- Category tags sourced from Task 2 preprocessing output for the selected scheme

## 3. Quantitative Results
### Side-by-side summary
{summary_df.to_markdown(index=False)}

### Per-category (Accuracy and Macro-F1)
{cat_cmp.to_markdown(index=False)}

### Best category gains for SOTA (F1 difference)
{best_cat_gain[['category', 'n_samples', 'optimized_f1', 'sota_f1', 'f1_gain']].head(5).to_markdown(index=False)}

### Largest drops for SOTA (F1 difference)
{worst_cat_gain[['category', 'n_samples', 'optimized_f1', 'sota_f1', 'f1_gain']].head(5).to_markdown(index=False)}

## 4. Visual Analysis
The following plots were generated and saved:
- overall_metrics_comparison.png
- per_category_accuracy.png
- per_category_f1.png
- confusion_matrices_side_by_side.png
- error_rate_by_category_heatmap.png

Interpretation guide:
- Overall metrics chart indicates global performance dominance.
- Per-category bars reveal category-specific robustness differences.
- Confusion matrices show which sentiment classes are most confused by each model.
- Error-rate heatmap highlights category-level failure concentration.

## 5. Conclusion
This benchmark provides a direct comparison between a task-optimized in-project model and a transferable SOTA transformer model.
Use the per-category and confusion analysis to decide whether to keep one global model, ensemble both, or route by category.
"""

report_path = TASK5_DIR / 'task5_results_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_md)

print('Saved structured markdown report to:', report_path)
print('\nPreview:\n')
print(report_md[:2000])
